# 02. Automated Quality Control

## Overview
This notebook implements an automated **Quality Control (QC)** pipeline. Retinal fundus images are often plagued by artifacts (blur, poor illumination, lens caps) that can destabilize model training. 
Instead of manual curation, we calculate a **Quality Score** based on **Vessel Density** (the visibility of retinal blood vessels after preprocessing) to automatically flag ungradable images.

## Objectives
-   Analyze the distribution of vessel density across the dataset (blur image QC)
-   Determine a statistical threshold for rejection (e.g., the 5th percentile).
-   Visually validate the threshold by comparing high-quality vs. low-quality samples.
-   Generate a curated manifest (`manifest_filtered.csv`) for robust model training.

In [ ]:
# --- Imports ---
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Add project root
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import PROCESSED_DATA_DIR, DATA_DIR, QUALITY_THRESHOLD
from src.data.preprocessing import ImagePreprocessor
from src.data.make_dataset import load_filtered_dataset
from src.visualization.plots import set_plot_style, plot_quality_score_distribution
from src.visualization.eda import load_and_show_quality_examples

# --- Configuration ---
set_plot_style()
preprocessor = ImagePreprocessor()

In [ ]:
# 2. Load Manifest
manifest_path = PROCESSED_DATA_DIR / "manifest.csv"

if manifest_path.exists():
    df = pd.read_csv(manifest_path)
    print(f"Loaded manifest with {len(df)} images.")
    
    # Check if scores exist
    if 'quality_score' not in df.columns:
        print("\nWARNING: 'quality_score' column missing!")
        print("Please run 'src/scripts/assess_quality.py' to compute scores first.")
    else:
        print("Quality scores found.")
        print(df['quality_score'].describe())
else:
    print("Manifest not found.")

In [ ]:
# 3. Distribution of Scores
if 'quality_score' in df.columns:
    threshold = plot_quality_score_distribution(df)
    plt.show()

In [ ]:
# 4. Visualize Low Quality Examples
if 'quality_score' in df.columns:
    load_and_show_quality_examples(df, preprocessor, threshold, quality_type="low")

In [ ]:
# 4b. Visualize High Quality Examples
if 'quality_score' in df.columns:
    load_and_show_quality_examples(df, preprocessor, threshold, quality_type="high")

In [ ]:
# 5. Connect to Pipeline Config
# Verify the config threshold matches our analysis.

CHOSEN_THRESHOLD = QUALITY_THRESHOLD

print(f"Configured Quality Threshold: {CHOSEN_THRESHOLD}")

# Verify logic using the loader
df_filtered_test = load_filtered_dataset(manifest_path, min_quality_score=CHOSEN_THRESHOLD)

print(f"\n--- Verification ---")
print(f"Original Count: {len(df)}")
print(f"Filtered Count: {len(df_filtered_test)}")
print(f"Removed: {len(df) - len(df_filtered_test)} images ({100 * (len(df) - len(df_filtered_test)) / len(df):.1f}%)")